# Parda — Phase 6, step 1: models for the browser (ONNX) + proof that nothing was lost

Exports YOLO and GLiNER v2 to ONNX, compresses GLiNER to 8 bits (the file the web app will download), then
re-runs **both benchmarks** with the converted models and compares them with the PyTorch v2 results.
Everything runs on CPU, like it will in a browser. OCR is reused from the v2 run (08b), so no OCR engine is needed.

**Settings:** Accelerator **None (CPU)** — free · Internet **On**
**Inputs (Add Input):** Your Work → **Parda** (Phase 1) · Your Work → the **08b** notebook (text model v2) ·
Datasets tab → `celeba-dataset`. **Secrets:** `GITHUB_TOKEN`, `HF_TOKEN`

Run cells 1–4 by hand (~5 min), then **Save Version → Save & Run All** (~1.5 h).


In [ ]:
# 1. Code + secrets
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GH = secrets.get_secret("GITHUB_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
REPO, CODE = "chetanjagan/Parda", "/tmp/Parda"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "-q", f"https://{GH}@github.com/{REPO}.git", CODE], check=True)
    subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "-q", f"https://{GH}@github.com/{REPO}.git", "main"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)


In [ ]:
# 2. Fonts (to regenerate the unseen test pages) + GLiNER, YOLO, ONNX tools
!bash scripts/setup_kaggle.sh
!pip install -q gliner ultralytics onnx onnxruntime
!python -c "import gliner, torch, onnx, onnxruntime, ultralytics; print('gliner', gliner.__version__, '| torch', torch.__version__, '| onnx', onnx.__version__, '| onnxruntime', onnxruntime.__version__)"


In [ ]:
# 3. Find the inputs: Phase 1 data, faces, the saved OCR and the PyTorch v2 reports (from 08b)
import json, shutil
from huggingface_hub import HfApi
from parda.fsutil import find_path
from parda.ocr.run_ocr import find_data, _n_images
from parda.vision.faces import FacePool, find_faces_dir
IN, W = "/kaggle/input", "/kaggle/working"
first = lambda *rels: next((h[0] for r in rels for h in [find_path(IN, r)] if h), None)
DATA = find_data()
FACES = find_faces_dir(IN, exclude=[DATA])
C_SEEN = first("cache_seen/ocr_easyocr.jsonl", "e2e_cache/ocr_easyocr.jsonl")
C_OOD = first("cache_ood/ocr_easyocr.jsonl", "ood_cache/ocr_easyocr.jsonl")
V2_SEEN, V2_OOD = first("v2_seen/report_e2e.json"), first("v2_ood/report_e2e.json")
HF_USER = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"]
GLINER_V2, YOLO_REPO = f"{HF_USER}/parda-gliner-v2", f"{HF_USER}/parda-yolo-v1"
for k, v in [("DATA", DATA), ("FACES", FACES), ("OCR seen", C_SEEN), ("OCR unseen", C_OOD),
             ("v2 report seen", V2_SEEN), ("v2 report unseen", V2_OOD), ("models", f"{GLINER_V2} | {YOLO_REPO}")]:
    print(f"{k:18s} {v}")
assert DATA and _n_images(DATA) == 30000, "attach your Parda (Phase 1) notebook"
assert FACES, "Add Input -> Datasets tab -> 'celeba-dataset'"
assert C_SEEN and C_OOD and V2_SEEN and V2_OOD, "attach the 08b notebook (its cache_seen, cache_ood, v2_seen, v2_ood)"
C_SEEN, C_OOD = os.path.dirname(C_SEEN), os.path.dirname(C_OOD)


In [ ]:
# 4. Tests (~1 min): every line ok, and the round trip must NOT say "skipped"
!python tests/test_export.py


In [ ]:
# 5. EXPORT: YOLO -> ONNX, GLiNER v2 -> ONNX (full precision + 8-bit), and check they find the same spans as
#    PyTorch on 200 real OCR page texts (~30 min on CPU)
!python -m parda.export.onnx_export --gliner {GLINER_V2} --yolo {YOLO_REPO} --out {W}/onnx --parity_ocr {C_SEEN}/ocr_tesseract.jsonl {C_OOD}/ocr_easyocr.jsonl --parity_n 100 2>&1 | grep -v -i warning | tail -15
summ = json.load(open(f"{W}/onnx/export_summary.json"))
print(json.dumps({"MB": {**summ["gliner"]["mb"], "yolo": summ["yolo"]["mb"]}, "exporter": summ["gliner"]["exporter"],
                  "same spans as PyTorch": {k: v["agreement_f1"] for k, v in summ["agreement"].items()}}, indent=2))


In [ ]:
# 6. YOLO: PyTorch vs ONNX on the 300 benchmark pages (~10 min on CPU)
from parda.pipeline.models import yolo_weights
PT = yolo_weights(YOLO_REPO)
!python -m parda.vision.yolo_data --data {DATA} --out {W}/yolo_bench --faces_dir {FACES} --only bench --workers 4 > /dev/null
!python -m parda.vision.evaluate_vis --yolo_data {W}/yolo_bench --systems yolo --weights {PT} --thresholds 0.25 --out {W}/vis_pt > /dev/null
!python -m parda.vision.evaluate_vis --yolo_data {W}/yolo_bench --systems yolo --weights {W}/onnx/yolo/parda-yolo.onnx --thresholds 0.25 --out {W}/vis_onnx > /dev/null
vp, vo = (json.load(open(f"{W}/{d}/report_vis.json"))["systems"][0] for d in ("vis_pt", "vis_onnx"))
print(f"YOLO fully redacted: PyTorch {vp['fully_redacted']:.1%} | ONNX {vo['fully_redacted']:.1%}   "
      f"mAP50: {vp['map50']:.3f} | {vo['map50']:.3f}")


In [ ]:
# 7. END TO END with the 8-bit GLiNER + ONNX YOLO: seen templates (OCR reused) (~15 min)
os.makedirs(f"{W}/cache_seen", exist_ok=True)
for f in ("ocr_tesseract.jsonl", "ocr_easyocr.jsonl"):
    shutil.copy(f"{C_SEEN}/{f}", f"{W}/cache_seen/{f}")
!python -m parda.pipeline.evaluate_e2e --data {DATA} --yolo_data {W}/yolo_bench --ocr tesseract,easyocr --gliner_ft {W}/onnx/gliner/model_int8.onnx --yolo {W}/onnx/yolo/parda-yolo.onnx --cache {W}/cache_seen --out {W}/onnx_seen --title "8-bit ONNX models — seen templates" 2>&1 | grep -v -i warning | tail -10


In [ ]:
# 8. END TO END with the 8-bit models: unseen layouts + phone photos (same 300 pages; OCR reused) (~15 min)
!python -m parda.synth.ood --n 300 --out {W}/ood --faces_dir {FACES} --workers 4 > /dev/null
os.makedirs(f"{W}/cache_ood", exist_ok=True)
for f in ("ocr_tesseract.jsonl", "ocr_easyocr.jsonl"):
    shutil.copy(f"{C_OOD}/{f}", f"{W}/cache_ood/{f}")
!python -m parda.pipeline.evaluate_e2e --data {W}/ood --ocr tesseract,easyocr --gliner_ft {W}/onnx/gliner/model_int8.onnx --yolo {W}/onnx/yolo/parda-yolo.onnx --cache {W}/cache_ood --out {W}/onnx_ood --title "8-bit ONNX models — unseen layouts + phone photos" 2>&1 | grep -v -i warning | tail -10


In [ ]:
# 9. PyTorch v2 vs 8-bit ONNX, side by side
from IPython.display import Markdown, display
pct = lambda v: "–" if v is None else f"{100 * v:.1f}%"
rows = ["| Benchmark | Configuration | All PII: PyTorch → ONNX 8-bit | Text | Pages with no leak | s/page (ONNX, CPU) |",
        "|---|---|---|---|---|---|"]
drops = []
for bench, v2p, onp in (("seen", V2_SEEN, f"{W}/onnx_seen/report_e2e.json"), ("unseen", V2_OOD, f"{W}/onnx_ood/report_e2e.json")):
    v2 = {r["config"]: r for r in json.load(open(v2p))["configs"]}
    on = {r["config"]: r for r in json.load(open(onp))["configs"]}
    for name in ("+ YOLO visual = Parda", "browser build: Tesseract only + YOLO"):
        a, b = v2.get(name, {}), on[name]
        if a:
            drops.append(a["fully_redacted"] - b["fully_redacted"])
        rows.append(f"| {bench} | {name} | {pct(a.get('fully_redacted'))} → **{pct(b['fully_redacted'])}** | "
                    f"{pct(a.get('text'))} → {pct(b['text'])} | {pct(a.get('pages_clean'))} → {pct(b['pages_clean'])} | {b['sec_per_page']:.2f} |")
sizes = f"GLiNER {summ['gliner']['mb'].get('fp32')} MB → **{summ['gliner']['mb'].get('int8')} MB** (8-bit) · YOLO {summ['yolo']['mb']} MB"
agree = ", ".join(f"{k}: {v['agreement_f1']:.1%}" for k, v in summ["agreement"].items())
verdict = f"Largest drop vs PyTorch: {100 * max(drops):.1f} points." if drops else ""
report = "\n".join(["### PyTorch v2 vs ONNX (the files the web app will use)", "", *rows, "",
                    f"Model sizes: {sizes}", f"Same PII spans as PyTorch on 200 real OCR texts: {agree}",
                    f"YOLO fully redacted: PyTorch {vp['fully_redacted']:.1%} → ONNX {vo['fully_redacted']:.1%}", verdict]) + "\n"
open(f"{W}/onnx_vs_pytorch.md", "w").write(report)
display(Markdown(report))


In [ ]:
# 10. Save the ONNX models to a private HF repo (the web app will load them from there later)
from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(f"{HF_USER}/parda-onnx-v1", private=True, exist_ok=True)
api.upload_folder(repo_id=f"{HF_USER}/parda-onnx-v1", folder_path=f"{W}/onnx", commit_message="GLiNER v2 (fp32 + int8) and YOLO11n ONNX")
print("uploaded to", f"https://huggingface.co/{HF_USER}/parda-onnx-v1")


In [ ]:
# 11. Collect results to commit to GitHub (download from the Output tab)
R = f"{W}/onnx_results"
os.makedirs(R, exist_ok=True)
for src, dst in [(f"{W}/onnx_vs_pytorch.md", "onnx_vs_pytorch.md"), (f"{W}/onnx/export_summary.json", "export_summary.json"),
                 (f"{W}/onnx_seen/report_e2e.md", "report_seen_onnx.md"), (f"{W}/onnx_seen/report_e2e.json", "report_seen_onnx.json"),
                 (f"{W}/onnx_ood/report_e2e.md", "report_unseen_onnx.md"), (f"{W}/onnx_ood/report_e2e.json", "report_unseen_onnx.json")]:
    if os.path.exists(src):
        shutil.copy(src, os.path.join(R, dst))
print(sorted(os.listdir(R)))
